# WikiStream Analytics Pipeline

## Stream Processing Pipeline with Wikimedia Recent Changes Data

Este notebook apresenta a construção de um pipeline de **stream processing** utilizando o stream público de mudanças recentes da Wikimedia.

A fonte utilizada é:

```text
https://stream.wikimedia.org/v2/stream/recentchange
```

O pipeline realiza ingestão de eventos contínuos, armazenamento dos dados brutos em JSON, processamento com Apache Spark Structured Streaming, validação, filtros, conversão para Parquet, agregações analíticas, agregação temporal com Window Function e proposta de deploy.

A execução foi estruturada de forma finita para fins acadêmicos, utilizando `trigger(availableNow=True)`. Ainda assim, o processamento é realizado com Spark Structured Streaming, por meio de `readStream`, `writeStream`, checkpoints e leitura incremental dos arquivos disponíveis.

## 1. Visão geral da solução

A solução utiliza o Databricks como plataforma de processamento, Apache Spark Structured Streaming como mecanismo de stream processing e Unity Catalog Volume como camada de armazenamento governada.

O objetivo é transformar eventos contínuos da Wikimedia em dados analíticos organizados em camadas:

```text
Wikimedia Recent Changes Stream
        ↓
Raw JSON Events
        ↓
Bronze Parquet
        ↓
Silver Validated Parquet
        ↓
Gold Aggregations and Window Outputs
```

Essa estrutura segue uma abordagem inspirada na arquitetura Medallion, favorecendo rastreabilidade, separação lógica das etapas e evolução para um pipeline operacional.

## 2. Criação do schema e do Unity Catalog Volume

A solução utiliza um Unity Catalog Volume como camada de armazenamento governada para suportar a organização dos dados em Raw, Bronze, Silver e Gold, além dos checkpoints do processamento streaming.

Essa abordagem favorece rastreabilidade, separação lógica das etapas do pipeline e maior alinhamento com práticas modernas de engenharia de dados em ambientes Databricks.

In [0]:
# Define os nomes dos objetos que serão usados no Unity Catalog.
# O catalog "workspace" será utilizado como camada principal.
# O schema organiza os objetos relacionados ao trabalho de Stream Processing.
# O volume será a área física/governada para armazenar arquivos do pipeline.
CATALOG_NAME = "workspace"
SCHEMA_NAME = "stream_processing_pipelines"
VOLUME_NAME = "wikimedia_stream_pipeline"

# Cria o schema caso ele ainda não exista.
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}
COMMENT 'Schema used for stream processing pipeline projects'
""")

# Cria o Unity Catalog Volume caso ele ainda não exista.
spark.sql(f"""
CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.{VOLUME_NAME}
COMMENT 'Volume used for the WikiStream Analytics Pipeline project'
""")

# Exibe os objetos configurados para facilitar a validação da execução.
print(f"Schema configured: {CATALOG_NAME}.{SCHEMA_NAME}")
print(f"Volume configured: {CATALOG_NAME}.{SCHEMA_NAME}.{VOLUME_NAME}")

Schema configured: workspace.stream_processing_pipelines
Volume configured: workspace.stream_processing_pipelines.wikimedia_stream_pipeline


In [0]:
# Exibe os metadados do volume criado no Unity Catalog (validação para confirmar que o volume existe e está disponível para uso no pipeline).
spark.sql(f"DESCRIBE VOLUME {CATALOG_NAME}.{SCHEMA_NAME}.{VOLUME_NAME}").show(truncate=False)

+-------------------------+---------+---------------------------+---------------------+----------------+-----------+---------------------------------------------------------+--------------+-----------------+
|name                     |catalog  |database                   |owner                |storage_location|volume_type|comment                                                  |securable_type|securable_kind   |
+-------------------------+---------+---------------------------+---------------------+----------------+-----------+---------------------------------------------------------+--------------+-----------------+
|wikimedia_stream_pipeline|workspace|stream_processing_pipelines|thatianemsb@gmail.com|                |MANAGED    |Volume used for the WikiStream Analytics Pipeline project|VOLUME        |VOLUME_DB_STORAGE|
+-------------------------+---------+---------------------------+---------------------+----------------+-----------+----------------------------------------------------

## 3. Configuração dos caminhos do pipeline

Os caminhos abaixo organizam os arquivos do pipeline em áreas específicas para ingestão, processamento, outputs analíticos e checkpoints.

Os checkpoints são fundamentais em pipelines de Structured Streaming, pois registram o progresso do processamento incremental e permitem maior controle sobre reprocessamentos.

In [0]:
# Todos os diretórios do pipeline serão criados dentro deste volume.
VOLUME_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}"

# Nesta área serão armazenados os eventos brutos recebidos do stream da Wikimedia em JSON Lines.
RAW_PATH = f"{VOLUME_PATH}/raw/events"

# Caminhos das camadas de saída do pipeline:
# Bronze: dados brutos convertidos para Parquet.
# Silver: dados validados, filtrados e enriquecidos.
# Gold: agregações analíticas e temporais.
BRONZE_OUTPUT_PATH = f"{VOLUME_PATH}/bronze/events_parquet"
SILVER_OUTPUT_PATH = f"{VOLUME_PATH}/silver/validated_events_parquet"
GOLD_AGG_OUTPUT_PATH = f"{VOLUME_PATH}/gold/aggregations_parquet"
GOLD_WINDOW_OUTPUT_PATH = f"{VOLUME_PATH}/gold/window_aggregations_parquet"

# Caminhos de checkpoint do Spark Structured Streaming (cada etapa streaming possui seu próprio checkpoint para controlar progresso e evitar reprocessamento indevido).
CHECKPOINT_BRONZE_PATH = f"{VOLUME_PATH}/checkpoints/bronze"
CHECKPOINT_SILVER_PATH = f"{VOLUME_PATH}/checkpoints/silver"
CHECKPOINT_WINDOW_PATH = f"{VOLUME_PATH}/checkpoints/window"

# Exibe os principais caminhos configurados.
print("Volume path:", VOLUME_PATH)
print("Raw path:", RAW_PATH)
print("Bronze output:", BRONZE_OUTPUT_PATH)
print("Silver output:", SILVER_OUTPUT_PATH)
print("Gold aggregations output:", GOLD_AGG_OUTPUT_PATH)
print("Gold window output:", GOLD_WINDOW_OUTPUT_PATH)

Volume path: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline
Raw path: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/raw/events
Bronze output: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/bronze/events_parquet
Silver output: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/silver/validated_events_parquet
Gold aggregations output: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/gold/aggregations_parquet
Gold window output: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/gold/window_aggregations_parquet


### Reprodutibilidade da execução

O notebook inclui uma opção de limpeza controlada dos diretórios de dados e checkpoints antes da execução.

Essa configuração permite executar a demonstração de forma isolada, evitando que resultados de testes anteriores sejam misturados aos resultados da rodada atual.

Quando a opção `RESET_PIPELINE_OUTPUTS` está ativada, o pipeline é reiniciado para gerar uma evidência limpa da execução.

In [0]:
# Controla se os outputs anteriores devem ser removidos antes de uma nova execução.
# Quando True, a execução começa limpa, sem misturar resultados de testes anteriores.
# Quando False, os dados e checkpoints são preservados para simular continuidade incremental entre execuções.
RESET_PIPELINE_OUTPUTS = True

# Lista de diretórios que podem ser limpos antes da execução.
paths_to_clean = [
    RAW_PATH,
    BRONZE_OUTPUT_PATH,
    SILVER_OUTPUT_PATH,
    GOLD_AGG_OUTPUT_PATH,
    GOLD_WINDOW_OUTPUT_PATH,
    CHECKPOINT_BRONZE_PATH,
    CHECKPOINT_SILVER_PATH,
    CHECKPOINT_WINDOW_PATH
]

# Remove os diretórios anteriores quando a limpeza controlada está ativada.
if RESET_PIPELINE_OUTPUTS:
    for path in paths_to_clean:
        try:
            dbutils.fs.rm(path, recurse=True)
            print(f"Removido: {path}")
        except Exception:
            print(f"Caminho não encontrado ou não removido: {path}")

    print("Outputs e checkpoints anteriores foram removidos.")
else:
    print("Outputs e checkpoints anteriores foram preservados.")

Removido: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/raw/events
Removido: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/bronze/events_parquet
Removido: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/silver/validated_events_parquet
Removido: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/gold/aggregations_parquet
Removido: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/gold/window_aggregations_parquet
Removido: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/checkpoints/bronze
Removido: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/checkpoints/silver
Removido: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/checkpoints/window
Outputs e checkpoints anteriores foram removidos.


In [0]:
# Lista de diretórios necessários para a execução do pipeline (esses diretórios organizam as áreas Raw, Bronze, Silver, Gold e os checkpoints do streaming).
paths = [
    RAW_PATH,
    BRONZE_OUTPUT_PATH,
    SILVER_OUTPUT_PATH,
    GOLD_AGG_OUTPUT_PATH,
    GOLD_WINDOW_OUTPUT_PATH,
    CHECKPOINT_BRONZE_PATH,
    CHECKPOINT_SILVER_PATH,
    CHECKPOINT_WINDOW_PATH
]

# Cria cada diretório dentro do Unity Catalog Volume.
for path in paths:
    dbutils.fs.mkdirs(path)

# Exibe confirmação da configuração dos diretórios.
print("Diretórios do pipeline configurados com sucesso.")
print("Volume path:", VOLUME_PATH)

Diretórios do pipeline configurados com sucesso.
Volume path: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline


In [0]:
# Evidência de que os diretórios foram criados corretamente
display(dbutils.fs.ls(VOLUME_PATH))

path,name,size,modificationTime
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/bronze/,bronze/,0,1777745901087
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/checkpoints/,checkpoints/,0,1777745901087
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/gold/,gold/,0,1777745901087
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/raw/,raw/,0,1777745901087
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/silver/,silver/,0,1777745901087


## 4. Fonte de dados: Wikimedia Recent Changes

A fonte escolhida é o stream público de mudanças recentes da Wikimedia, que publica eventos continuamente à medida que alterações ocorrem em projetos como Wikipedia, Wikidata, Wikimedia Commons e outros.

O endpoint disponibiliza eventos em tempo real utilizando o padrão **Server-Sent Events (SSE)**.

Neste notebook, a conexão SSE é consumida com a biblioteca `requests`, mantendo a conexão HTTP aberta com `stream=True` e processando as linhas recebidas por meio de `response.iter_lines()`.

As mensagens de evento são identificadas pelo prefixo `data:`, convertidas de JSON para dicionários Python e armazenadas na área Raw em formato JSON Lines.

In [0]:
# Import das bibliotecas utilizadas na ingestão.
import json
import time
import requests
from datetime import datetime, timezone

# Endpoint público da Wikimedia que publica eventos recentes em tempo real.
STREAM_URL = "https://stream.wikimedia.org/v2/stream/recentchange"


def ingest_wikimedia_stream(duration_seconds=60, max_events=300):
    """
    Consome o stream Wikimedia Recent Changes e grava os eventos em JSON Lines
    na camada Raw do Unity Catalog Volume.

    Parâmetros:
    duration_seconds: tempo máximo de ingestão em segundos.
    max_events: quantidade máxima de eventos a serem capturados.
    """

    start_time = time.time()
    event_count = 0

    output_file = (
        f"{RAW_PATH}/wikimedia_events_"
        f"{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}.json"
    )

    headers = {
        "User-Agent": "WikiStreamAnalyticsPipeline/1.0"
    }

    # Abre uma conexão HTTP contínua com a fonte de streaming.
    # O parâmetro stream=True mantém a conexão aberta para leitura incremental das mensagens.
    with requests.get(STREAM_URL, headers=headers, stream=True, timeout=120) as response:

        response.raise_for_status()

        with open(output_file, "w", encoding="utf-8") as file:

            for line in response.iter_lines(decode_unicode=True):

                if time.time() - start_time > duration_seconds:
                    break

                if event_count >= max_events:
                    break

                if not line or not line.startswith("data: "):
                    continue

                payload = line.replace("data: ", "", 1)

                try:
                    data = json.loads(payload)
                    data["_ingestion_timestamp"] = datetime.now(timezone.utc).isoformat()
                    file.write(json.dumps(data, ensure_ascii=False) + "\n")
                    event_count += 1

                except json.JSONDecodeError:
                    continue

    # Exibe o resumo da ingestão.
    print("Ingestão concluída.")
    print(f"Eventos ingeridos: {event_count}")
    print(f"Arquivo gerado: {output_file}")

In [0]:
# Executa a ingestão por até 60 segundos ou até capturar 300 eventos.
ingest_wikimedia_stream(duration_seconds=60, max_events=300)

Ingestão concluída.
Eventos ingeridos: 300
Arquivo gerado: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/raw/events/wikimedia_events_20260502_181824.json


In [0]:
# Lista os arquivos gerados na camada Raw.
display(dbutils.fs.ls(RAW_PATH))

path,name,size,modificationTime
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/raw/events/wikimedia_events_20260502_181824.json,wikimedia_events_20260502_181824.json,452167,1777745914000


## 5. Definição do schema dos eventos

Para processar os eventos com Spark Structured Streaming, o schema dos principais campos é declarado explicitamente.

Essa prática evita inferência automática em dados contínuos, melhora a estabilidade do pipeline e facilita a aplicação de validações e transformações posteriores.

In [0]:
# Importa os tipos necessários para definir explicitamente o schema dos eventos.
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    LongType,
    IntegerType
)

wikimedia_schema = StructType([
    StructField("$schema", StringType(), True),
    StructField("meta", StructType([
        StructField("uri", StringType(), True),
        StructField("request_id", StringType(), True),
        StructField("id", StringType(), True),
        StructField("dt", StringType(), True),
        StructField("domain", StringType(), True),
        StructField("stream", StringType(), True),
        StructField("topic", StringType(), True),
        StructField("partition", IntegerType(), True),
        StructField("offset", LongType(), True)
    ]), True),
    StructField("id", LongType(), True),
    StructField("type", StringType(), True),
    StructField("namespace", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("title_url", StringType(), True),
    StructField("comment", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("user", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("patrolled", BooleanType(), True),
    StructField("length", StructType([
        StructField("old", LongType(), True),
        StructField("new", LongType(), True)
    ]), True),
    StructField("revision", StructType([
        StructField("old", LongType(), True),
        StructField("new", LongType(), True)
    ]), True),
    StructField("server_url", StringType(), True),
    StructField("server_name", StringType(), True),
    StructField("server_script_path", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("parsedcomment", StringType(), True),
    StructField("_ingestion_timestamp", StringType(), True)
])

## 6. Camada Bronze: leitura streaming e conversão para Parquet

A camada Bronze representa a primeira persistência dos eventos brutos após a ingestão.

Nesta etapa, o Spark lê os arquivos JSON da área Raw utilizando `readStream`, processa os arquivos de forma incremental e grava a saída em Parquet utilizando `writeStream`.

Essa etapa atende ao requisito central do trabalho: realizar a ingestão de dados e gerar output em outro formato, neste caso de JSON para Parquet.

In [0]:
# Cria um DataFrame de streaming a partir dos arquivos JSON da camada Raw.
raw_stream_df = (
    spark.readStream
    .schema(wikimedia_schema)
    .option("maxFilesPerTrigger", 10)
    .json(RAW_PATH)
)

# Confirma que o DataFrame foi criado em modo streaming.
print("Is raw_stream_df streaming?", raw_stream_df.isStreaming)

# Exibe o schema interpretado pelo Spark.
raw_stream_df.printSchema()

Is raw_stream_df streaming? True
root
 |-- $schema: string (nullable = true)
 |-- meta: struct (nullable = true)
 |    |-- uri: string (nullable = true)
 |    |-- request_id: string (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- dt: string (nullable = true)
 |    |-- domain: string (nullable = true)
 |    |-- stream: string (nullable = true)
 |    |-- topic: string (nullable = true)
 |    |-- partition: integer (nullable = true)
 |    |-- offset: long (nullable = true)
 |-- id: long (nullable = true)
 |-- type: string (nullable = true)
 |-- namespace: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- title_url: string (nullable = true)
 |-- comment: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- user: string (nullable = true)
 |-- bot: boolean (nullable = true)
 |-- minor: boolean (nullable = true)
 |-- patrolled: boolean (nullable = true)
 |-- length: struct (nullable = true)
 |    |-- old: long (nullable = true)
 |    |-- ne

In [0]:
# Grava os eventos brutos da Raw em formato Parquet na camada Bronze.
bronze_query = (
    raw_stream_df.writeStream
    .format("parquet")
    .option("checkpointLocation", CHECKPOINT_BRONZE_PATH)
    .option("path", BRONZE_OUTPUT_PATH)
    .outputMode("append")
    .trigger(availableNow=True)
    .start()
)

bronze_query.awaitTermination()

print("Bronze streaming query completed.")
print("Query status:", bronze_query.status)
print("Last progress:", bronze_query.lastProgress)

Bronze streaming query completed.
Query status: {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}
Last progress: {'id': 'bdbd0ef8-8982-4686-a8a5-e5e04d286dc7', 'runId': 'f7d5ccee-2960-4254-8e14-2d7743debe59', 'name': None, 'timestamp': '2026-05-02T18:18:36.556Z', 'batchId': 0, 'batchDuration': 1242, 'durationMs': {'triggerExecution': 1242, 'queryPlanning': 30, 'collectSourceMetrics': 1, 'getBatch': 166, 'commitOffsets': 67, 'addBatch': 832, 'latestOffset': 71, 'walCommit': 146}, 'eventTime': {}, 'stateOperators': [], 'sources': [{'description': 'FileStreamSource[dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/raw/events]', 'startOffset': None, 'endOffset': '{"logOffset":0}', 'latestOffset': None, 'numInputRows': 300, 'inputRowsPerSecond': 0.0, 'processedRowsPerSecond': 241.54589371980677, 'metrics': {}}], 'sink': {'description': 'FileSink[/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/bronze/events_parquet

In [0]:
bronze_df = spark.read.parquet(BRONZE_OUTPUT_PATH)

# Exibe uma amostra dos registros da Bronze.
display(bronze_df.limit(20))

# Exibe a quantidade total de registros gravados na Bronze.
print(f"Total records in Bronze: {bronze_df.count()}")

$schema,meta,id,type,namespace,title,title_url,comment,timestamp,user,bot,minor,patrolled,length,revision,server_url,server_name,server_script_path,wiki,parsedcomment,_ingestion_timestamp
/mediawiki/recentchange/1.0.0,"List(https://commons.wikimedia.org/wiki/File:Working_Day_(46392217315).jpg, 541996e5-704e-4400-8f47-54b7254863fc, 527cb7b8-de88-4076-957c-af46da07aced, 2026-05-02T18:18:25.466Z, commons.wikimedia.org, mediawiki.recentchange, eqiad.mediawiki.recentchange, 0, 6090052930)",3304024585,edit,6,File:Working Day (46392217315).jpg,https://commons.wikimedia.org/wiki/File:Working_Day_(46392217315).jpg,"/* wbsetclaim-create:1||1 */ [[d:Special:EntityPage/P1163]]: image/jpeg, BOT - Adding [[Commons:Structured data|structured data]] based on file information: media type",1777745904,Emijrpbot,true,false,true,"List(5591, 5944)","List(895936482, 1207244471)",https://commons.wikimedia.org,commons.wikimedia.org,/w,commonswiki,"‎Created claim: media type (P1163): image/jpeg, BOT - Adding structured data based on file information: media type",2026-05-02T18:18:25.969530+00:00
/mediawiki/recentchange/1.0.0,"List(https://fi.wikipedia.org/wiki/Luokka:Led_Zeppelinin_konserttikiertueet, 413d04da-d277-443c-bcc9-c4447372bb3a, 5edcbd8c-e523-4967-986e-217e87b7c661, 2026-05-02T18:18:25.499Z, fi.wikipedia.org, mediawiki.recentchange, eqiad.mediawiki.recentchange, 0, 6090052931)",56348659,categorize,14,Luokka:Led Zeppelinin konserttikiertueet,https://fi.wikipedia.org/wiki/Luokka:Led_Zeppelinin_konserttikiertueet,[[:Led Zeppelin European Tour 1970]] lisätty luokkaan,1777745904,Thi,false,null,null,null,null,https://fi.wikipedia.org,fi.wikipedia.org,/w,fiwiki,Led Zeppelin European Tour 1970 lisätty luokkaan,2026-05-02T18:18:25.969663+00:00
/mediawiki/recentchange/1.0.0,"List(https://www.wikidata.org/wiki/Q8022159, 287e3762-953d-4373-9432-df2b16e89dd2, c0c5376c-64c3-4760-a68f-45e2844208fe, 2026-05-02T18:18:25.580Z, www.wikidata.org, mediawiki.recentchange, eqiad.mediawiki.recentchange, 0, 6090052932)",2563972251,edit,0,Q8022159,https://www.wikidata.org/wiki/Q8022159,"/* wbsetqualifier-add:1| */ [[Property:P1540]]: 5,989, #quickstatements; #temporary_batch_1777743330157",1777745905,Steinsky,false,false,true,"List(110800, 111067)","List(2486621685, 2486621695)",https://www.wikidata.org,www.wikidata.org,/w,wikidatawiki,"‎Added qualifier: male population (P1540): 5,989, #quickstatements; #temporary_batch_1777743330157",2026-05-02T18:18:25.969740+00:00
/mediawiki/recentchange/1.0.0,"List(https://en.wikisource.org/wiki/Page:%E0%B8%9E%E0%B8%A3%E0%B8%9A_%E0%B8%A3%E0%B8%B2%E0%B8%8A%E0%B8%9A%E0%B8%B1%E0%B8%93%E0%B8%91%E0%B8%B4%E0%B8%95%E0%B8%AF_%E0%B9%92%E0%B9%95%E0%B9%94%E0%B9%94.pdf/3, cfde90f2-6cac-4688-8e40-eca6a4033092, 6d21d16b-c27d-4de9-ba58-262aac871e47, 2026-05-02T18:18:25.576Z, en.wikisource.org, mediawiki.recentchange, eqiad.mediawiki.recentchange, 0, 6090052933)",29498171,edit,104,Page:พรบ ราชบัณฑิตฯ ๒๕๔๔.pdf/3,https://en.wikisource.org/wiki/Page:%E0%B8%9E%E0%B8%A3%E0%B8%9A_%E0%B8%A3%E0%B8%B2%E0%B8%8A%E0%B8%9A%E0%B8%B1%E0%B8%93%E0%B8%91%E0%B8%B4%E0%B8%95%E0%B8%AF_%E0%B9%92%E0%B9%95%E0%B9%94%E0%B9%94.pdf/3,,1777745904,Flamevine,false,false,false,"List(2287, 2249)","List(15923699, 15923718)",https://en.wikisource.org,en.wikisource.org,/w,enwikisource,,2026-05-02T18:18:25.969832+00:00
/mediawiki/recentchange/1.0.0,"List(https://commons.wikimedia.org/wiki/File:Klosterh%C3%BCtte_im_Neuhauser_Forst.jpg, 154b7532-7f8f-4bc6-9c31-f9e3be63da12, c293134d-492a-4b96-96ff-1c868be36190, 2026-05-02T18:18:25.605Z, commons.wikimedia.org, mediawiki.recentchange, eqiad.mediawiki.recentchange, 0, 6090052934)",3304024586,edit,6,File:Klosterhütte im Neuhauser Forst.jpg,https://commons.wikimedia.org/wiki/File:Klosterh%C3%BCtte_im_Neuhauser_Forst.jpg,"/* wbeditentity-statements-multiple-properties-add:0||5 */ BOT - Adding [[Commons:Structured data|structured data]] based on file information: exposure time, f-number, focal length, iso speed, media type",1777745904,Emijrpbo

Total records in Bronze: 300


## 7. Camada Silver: validação e filtros

A camada Silver é construída a partir da camada Bronze, mantendo a lógica Medallion do pipeline.

Nesta etapa, os eventos já persistidos em Parquet na Bronze são lidos novamente em modo streaming e submetidos a regras de validação, filtro e enriquecimento.

As regras aplicadas são:

- manter registros com `id` preenchido;
- manter registros com `type` preenchido;
- manter registros com `wiki` preenchido;
- manter registros com `title` preenchido;
- considerar apenas eventos dos tipos `edit`, `new`, `log` e `categorize`;
- converter o timestamp Unix para um campo temporal legível;
- calcular a diferença de tamanho da página quando essa informação estiver disponível.

Essa etapa demonstra a aplicação de validação e transformação dos dados dentro do pipeline de streaming.

In [0]:
# Importa funções Spark usadas nas regras de validação e enriquecimento da camada Silver.
from pyspark.sql.functions import (
    col,
    from_unixtime,
    to_timestamp,
    current_timestamp,
    when
)

# Lê a camada Bronze em modo streaming (Raw JSON → Bronze Parquet → Silver validada).
bronze_stream_df = (
    spark.readStream
    .schema(wikimedia_schema)
    .option("maxFilesPerTrigger", 10)
    .parquet(BRONZE_OUTPUT_PATH)
)

# Confirma que a leitura da Bronze está em modo streaming.
print("Is bronze_stream_df streaming?", bronze_stream_df.isStreaming)

# Aplica validações, filtros e enriquecimentos para construir a camada Silver.
validated_stream_df = (
    bronze_stream_df
    .filter(col("id").isNotNull())
    .filter(col("type").isNotNull())
    .filter(col("wiki").isNotNull())
    .filter(col("title").isNotNull())
    .filter(col("type").isin("edit", "new", "log", "categorize"))
    .withColumn("event_timestamp", to_timestamp(from_unixtime(col("timestamp"))))
    .withColumn("ingestion_timestamp", to_timestamp(col("_ingestion_timestamp")))
    .withColumn(
        "length_delta",
        when(
            col("length.new").isNotNull() & col("length.old").isNotNull(),
            col("length.new") - col("length.old")
        ).otherwise(None)
    )
    .withColumn("processed_at", current_timestamp())
    .select(
        "id",
        "type",
        "wiki",
        "title",
        "user",
        "bot",
        "minor",
        "server_name",
        "event_timestamp",
        "ingestion_timestamp",
        "length_delta",
        "processed_at"
    )
)

# Confirma que o DataFrame validado continua sendo streaming.
print("Is validated_stream_df streaming?", validated_stream_df.isStreaming)

Is bronze_stream_df streaming? True
Is validated_stream_df streaming? True


In [0]:
# Grava a camada Silver em Parquet.
silver_query = (
    validated_stream_df.writeStream
    .format("parquet")
    .option("checkpointLocation", CHECKPOINT_SILVER_PATH)
    .option("path", SILVER_OUTPUT_PATH)
    .outputMode("append")
    .trigger(availableNow=True)
    .start()
)

silver_query.awaitTermination()

print("Silver streaming query completed.")
print("Query status:", silver_query.status)
print("Last progress:", silver_query.lastProgress)

Silver streaming query completed.
Query status: {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}
Last progress: {'id': 'd413e5d6-68f9-4040-881e-1db9361eef2e', 'runId': '594c87dd-e3c4-481f-b52f-ba95c9ada2c9', 'name': None, 'timestamp': '2026-05-02T18:18:42.330Z', 'batchId': 0, 'batchDuration': 995, 'durationMs': {'triggerExecution': 995, 'queryPlanning': 54, 'collectSourceMetrics': 0, 'getBatch': 164, 'commitOffsets': 72, 'addBatch': 538, 'latestOffset': 81, 'walCommit': 168}, 'eventTime': {}, 'stateOperators': [], 'sources': [{'description': 'FileStreamSource[dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/bronze/events_parquet]', 'startOffset': None, 'endOffset': '{"logOffset":0}', 'latestOffset': None, 'numInputRows': 298, 'inputRowsPerSecond': 0.0, 'processedRowsPerSecond': 299.49748743718595, 'metrics': {}}], 'sink': {'description': 'FileSink[/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/silver/valid

In [0]:
# Lê a camada Silver como batch para inspeção e validação dos dados processados.
silver_df = spark.read.parquet(SILVER_OUTPUT_PATH)

# Exibe uma amostra dos registros validados.
display(silver_df.limit(20))

# Exibe o total de registros disponíveis na Silver.
print(f"Total records in Silver: {silver_df.count()}")

id,type,wiki,title,user,bot,minor,server_name,event_timestamp,ingestion_timestamp,length_delta,processed_at
3304024585,edit,commonswiki,File:Working Day (46392217315).jpg,Emijrpbot,true,false,commons.wikimedia.org,2026-05-02T18:18:24.000Z,2026-05-02T18:18:25.969Z,353,2026-05-02T18:18:42.411Z
56348659,categorize,fiwiki,Luokka:Led Zeppelinin konserttikiertueet,Thi,false,null,fi.wikipedia.org,2026-05-02T18:18:24.000Z,2026-05-02T18:18:25.969Z,null,2026-05-02T18:18:42.411Z
2563972251,edit,wikidatawiki,Q8022159,Steinsky,false,false,www.wikidata.org,2026-05-02T18:18:25.000Z,2026-05-02T18:18:25.969Z,267,2026-05-02T18:18:42.411Z
29498171,edit,enwikisource,Page:พรบ ราชบัณฑิตฯ ๒๕๔๔.pdf/3,Flamevine,false,false,en.wikisource.org,2026-05-02T18:18:24.000Z,2026-05-02T18:18:25.969Z,-38,2026-05-02T18:18:42.411Z
3304024586,edit,commonswiki,File:Klosterhütte im Neuhauser Forst.jpg,Emijrpbot,true,false,commons.wikimedia.org,2026-05-02T18:18:24.000Z,2026-05-02T18:18:25.969Z,1975,2026-05-02T18:18:42.411Z
56348660,categorize,fiwiki,Luokka:Led Zeppelin,Thi,false,null,fi.wikipedia.org,2026-05-02T18:18:24.000Z,2026-05-02T18:18:25.969Z,null,2026-05-02T18:18:42.411Z
8806081,new,zhwikisource,某某金融有限公司与薛某建金融借款合同纠纷一审民事判决书,SuperGrey-bot,true,false,zh.wikisource.org,2026-05-02T18:18:25.000Z,2026-05-02T18:18:25.970Z,null,2026-05-02T18:18:42.411Z
3304024587,categorize,commonswiki,"Category:Streets in the Czech Republic named after cities, villages and districts",Palickap,false,null,commons.wikimedia.org,2026-05-02T18:18:24.000Z,2026-05-02T18:18:25.970Z,null,2026-05-02T18:18:42.411Z
3501493,log,enwikinews,User talk:Thewriter1711,Zabe,false,null,en.wikinews.org,2026-05-02T18:18:25.000Z,2026-05-02T18:18:25.970Z,null,2026-05-02T18:18:42.411Z
2022880474,edit,enwiki,Neuromuscular-blocking drug,Solasoliloquy,false,true,en.wikipedia.org,2026-05-02T18:18:22.000Z,2026-05-02T18:18:25.981Z,31,2026-05-02T18:18:42.411Z


Total records in Silver: 298


In [0]:
# Calcula a distribuição dos eventos por tipo.
display(
    silver_df
    .groupBy("type")
    .count()
    .orderBy("count", ascending=False)
)

type,count
categorize,148
edit,120
log,18
new,12


## 8. Camada Gold: agregações analíticas

A camada Gold transforma os dados validados da Silver em indicadores analíticos.

As agregações abaixo demonstram como eventos de streaming podem ser convertidos em métricas úteis para análise, como volume de eventos por wiki, distribuição por tipo de evento e usuários mais ativos.

In [0]:
from pyspark.sql.functions import count, sum as spark_sum

# Agrega os eventos por wiki/projeto Wikimedia.
events_by_wiki_df = (
    silver_df
    .groupBy("wiki")
    .agg(
        count("*").alias("total_events"),
        spark_sum(when(col("bot") == True, 1).otherwise(0)).alias("bot_events"),
        spark_sum(when(col("bot") == False, 1).otherwise(0)).alias("human_events")
    )
    .orderBy(col("total_events").desc())
)

# Exibe o ranking de wikis com maior volume de eventos.
display(events_by_wiki_df)

wiki,total_events,bot_events,human_events
commonswiki,103,81,22
zhwikisource,40,40,0
cewiki,32,32,0
enwiktionary,24,23,1
wikidatawiki,19,3,16
enwiki,19,1,18
enwikinews,10,0,10
ruwikinews,7,7,0
nlwiktionary,7,0,7
frwiki,5,2,3


In [0]:
# Agrega os eventos por tipo de alteração.
events_by_type_df = (
    silver_df
    .groupBy("type")
    .agg(count("*").alias("total_events"))
    .orderBy(col("total_events").desc())
)

# Exibe a distribuição agregada por tipo de evento.
display(events_by_type_df)

type,total_events
categorize,148
edit,120
log,18
new,12


In [0]:
# Identifica os usuários com maior quantidade de eventos no período capturado.
top_users_df = (
    silver_df
    .groupBy("user")
    .agg(count("*").alias("total_events"))
    .orderBy(col("total_events").desc())
)

# Exibe os 20 usuários mais ativos.
display(top_users_df.limit(20))

user,total_events
SuperGrey-bot,40
CheWikibot,35
DPLA bot,30
FenaBot,23
Doc James,17
SchlurcherBot,16
Emijrpbot,15
OptimusPrimeBot,11
Zabe,10
Kvdrgeus,7


In [0]:
# Grava as agregações Gold em formato Parquet.
events_by_wiki_df.write.mode("overwrite").parquet(f"{GOLD_AGG_OUTPUT_PATH}/events_by_wiki")
events_by_type_df.write.mode("overwrite").parquet(f"{GOLD_AGG_OUTPUT_PATH}/events_by_type")
top_users_df.write.mode("overwrite").parquet(f"{GOLD_AGG_OUTPUT_PATH}/top_users")

# Confirma a gravação das agregações analíticas.
print("Gold aggregations written in Parquet format.")

Gold aggregations written in Parquet format.


## 9. Agregação temporal com Window Function

Nesta etapa, é aplicada uma agregação temporal sobre os dados validados da camada Silver.

A Silver é lida em modo streaming e os eventos são agrupados por wiki em janelas temporais. Essa abordagem demonstra uma análise típica de stream processing, útil para monitoramento de volume, detecção de picos e acompanhamento de atividade em tempo quase real.

A utilização de `readStream`, `withWatermark`, `window`, `writeStream` e checkpoints evidencia que a etapa é executada com Spark Structured Streaming.

In [0]:
from pyspark.sql.functions import window, count, col
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    LongType,
    TimestampType
)

# Define explicitamente o schema da camada Silver para leitura em modo streaming.
silver_schema = StructType([
    StructField("id", LongType(), True),
    StructField("type", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("title", StringType(), True),
    StructField("user", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("server_name", StringType(), True),
    StructField("event_timestamp", TimestampType(), True),
    StructField("ingestion_timestamp", TimestampType(), True),
    StructField("length_delta", LongType(), True),
    StructField("processed_at", TimestampType(), True)
])

silver_stream_df = (
    spark.readStream
    .schema(silver_schema)
    .option("maxFilesPerTrigger", 10)
    .parquet(SILVER_OUTPUT_PATH)
)

# Cria a agregação temporal com Window Function (os eventos são agrupados por wiki em janelas de 30 segundos).
window_stream_df = (
    silver_stream_df
    .filter(col("event_timestamp").isNotNull())
    .withWatermark("event_timestamp", "0 seconds")
    .groupBy(
        window(col("event_timestamp"), "30 seconds"),
        col("wiki")
    )
    .agg(count("*").alias("total_events"))
)

print("Is silver_stream_df streaming?", silver_stream_df.isStreaming)
print("Is window_stream_df streaming?", window_stream_df.isStreaming)

Is silver_stream_df streaming? True
Is window_stream_df streaming? True


In [0]:
# Grava o resultado da agregação temporal em Parquet.
window_query = (
    window_stream_df.writeStream
    .format("parquet")
    .option("checkpointLocation", CHECKPOINT_WINDOW_PATH)
    .option("path", GOLD_WINDOW_OUTPUT_PATH)
    .outputMode("append")
    .trigger(availableNow=True)
    .start()
)

window_query.awaitTermination()

# Exibe informações de status e progresso da query de Window.
print("Window streaming query completed.")
print("Query status:", window_query.status)
print("Last progress:", window_query.lastProgress)

Window streaming query completed.
Query status: {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}
Last progress: {'id': 'a9abf8dc-d6ab-4786-af0a-4566655067ae', 'runId': '7b541d98-3e00-4b49-8e70-fc458db8551b', 'name': None, 'timestamp': '2026-05-02T18:19:06.210Z', 'batchId': 1, 'batchDuration': 3907, 'durationMs': {'triggerExecution': 3907, 'queryPlanning': 40, 'collectSourceMetrics': 0, 'getBatch': 0, 'commitOffsets': 90, 'addBatch': 3692, 'latestOffset': 0, 'walCommit': 168}, 'eventTime': {'watermark': '2026-05-02T18:18:32.000Z'}, 'stateOperators': [{'operatorName': 'stateStoreSave', 'numRowsTotal': 13, 'numRowsUpdated': 0, 'allUpdatesTimeMs': 14, 'numRowsRemoved': 27, 'allRemovalsTimeMs': 1038, 'commitTimeMs': 10085, 'memoryUsedBytes': 269301344, 'numRowsDroppedByWatermark': 0, 'numShufflePartitions': 200, 'numStateStoreInstances': 200, 'customMetrics': {'rocksdbNumSnapshotsAutoRepaired': 0, 'rocksdbTotalBytesWrittenByFlush': 0, 'rocksdbGetLatency': 0, 'rocks

In [0]:
# Lê o resultado da agregação temporal gravado em Parquet.
window_results_df = spark.read.parquet(GOLD_WINDOW_OUTPUT_PATH)

display(
    window_results_df
    .orderBy(col("window.start").desc(), col("total_events").desc())
)

# Confirma a geração da saída da Window.
print("Window aggregation written in Parquet format.")

window,wiki,total_events
"List(2026-05-02T18:18:00.000Z, 2026-05-02T18:18:30.000Z)",commonswiki,82
"List(2026-05-02T18:18:00.000Z, 2026-05-02T18:18:30.000Z)",cewiki,26
"List(2026-05-02T18:18:00.000Z, 2026-05-02T18:18:30.000Z)",zhwikisource,25
"List(2026-05-02T18:18:00.000Z, 2026-05-02T18:18:30.000Z)",enwiktionary,23
"List(2026-05-02T18:18:00.000Z, 2026-05-02T18:18:30.000Z)",enwiki,18
"List(2026-05-02T18:18:00.000Z, 2026-05-02T18:18:30.000Z)",wikidatawiki,12
"List(2026-05-02T18:18:00.000Z, 2026-05-02T18:18:30.000Z)",nlwiktionary,7
"List(2026-05-02T18:18:00.000Z, 2026-05-02T18:18:30.000Z)",enwikinews,7
"List(2026-05-02T18:18:00.000Z, 2026-05-02T18:18:30.000Z)",ruwikinews,5
"List(2026-05-02T18:18:00.000Z, 2026-05-02T18:18:30.000Z)",frwiki,3


Window aggregation written in Parquet format.


## 10. Evidências técnicas de streaming

Além das saídas geradas pelo pipeline, esta seção apresenta evidências técnicas de que o processamento foi executado com Spark Structured Streaming.

As verificações abaixo demonstram que os DataFrames principais foram criados em modo streaming e que as queries utilizaram checkpoints para controle incremental do processamento.

In [0]:
# Consolida as principais evidências técnicas de que o pipeline foi executado com Spark Structured Streaming.
print("Streaming evidence")
print("------------------")

# Verifica se os DataFrames principais foram criados em modo streaming.
print("Raw DataFrame is streaming:", raw_stream_df.isStreaming)
print("Bronze DataFrame is streaming:", bronze_stream_df.isStreaming)
print("Validated Silver DataFrame is streaming:", validated_stream_df.isStreaming)
print("Silver Stream DataFrame is streaming:", silver_stream_df.isStreaming)
print("Window DataFrame is streaming:", window_stream_df.isStreaming)

# Exibe os caminhos de checkpoint usados por cada etapa streaming.
print("\nCheckpoint paths")
print("----------------")
print("Bronze checkpoint:", CHECKPOINT_BRONZE_PATH)
print("Silver checkpoint:", CHECKPOINT_SILVER_PATH)
print("Window checkpoint:", CHECKPOINT_WINDOW_PATH)

Streaming evidence
------------------
Raw DataFrame is streaming: True
Bronze DataFrame is streaming: True
Validated Silver DataFrame is streaming: True
Silver Stream DataFrame is streaming: True
Window DataFrame is streaming: True

Checkpoint paths
----------------
Bronze checkpoint: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/checkpoints/bronze
Silver checkpoint: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/checkpoints/silver
Window checkpoint: /Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/checkpoints/window


## 11. Evidências dos outputs gerados

Os comandos abaixo evidenciam os arquivos criados em cada etapa do pipeline dentro do Unity Catalog Volume.

Essa verificação confirma a materialização das áreas Raw, Bronze, Silver, Gold e Window Aggregations.

In [0]:
# Exibe os arquivos brutos capturados na camada Raw.
print("Raw files")
display(dbutils.fs.ls(RAW_PATH))

Raw files


path,name,size,modificationTime
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/raw/events/wikimedia_events_20260502_181824.json,wikimedia_events_20260502_181824.json,452167,1777745914000


In [0]:
# Exibe os arquivos Parquet gerados na camada Bronze.
print("Bronze output")
display(dbutils.fs.ls(BRONZE_OUTPUT_PATH))

Bronze output


path,name,size,modificationTime
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/bronze/events_parquet/_spark_metadata/,_spark_metadata/,0,1777745953440
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/bronze/events_parquet/part-00000-b174cbcd-9e49-4b61-95b5-8daf0974a086.c000.snappy.parquet,part-00000-b174cbcd-9e49-4b61-95b5-8daf0974a086.c000.snappy.parquet,90857,1777745918000


In [0]:
# Exibe os arquivos Parquet gerados na camada Silver.
print("Silver output")
display(dbutils.fs.ls(SILVER_OUTPUT_PATH))

Silver output


path,name,size,modificationTime
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/silver/validated_events_parquet/_spark_metadata/,_spark_metadata/,0,1777745954391
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/silver/validated_events_parquet/part-00000-8846a3a1-4c1e-4922-8c74-b669a5cc2011.c000.snappy.parquet,part-00000-8846a3a1-4c1e-4922-8c74-b669a5cc2011.c000.snappy.parquet,17444,1777745923000


In [0]:
# Exibe as subpastas das agregações Gold.
print("Gold aggregations output")
display(dbutils.fs.ls(GOLD_AGG_OUTPUT_PATH))

Gold aggregations output


path,name,size,modificationTime
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/gold/aggregations_parquet/events_by_type/,events_by_type/,0,1777745955374
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/gold/aggregations_parquet/events_by_wiki/,events_by_wiki/,0,1777745955374
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/gold/aggregations_parquet/top_users/,top_users/,0,1777745955374


In [0]:
# Exibe os arquivos da agregação temporal com Window Function.
print("Gold window output")
display(dbutils.fs.ls(GOLD_WINDOW_OUTPUT_PATH))

Gold window output


path,name,size,modificationTime
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/gold/window_aggregations_parquet/_spark_metadata/,_spark_metadata/,0,1777745956316
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/gold/window_aggregations_parquet/part-00000-1e340abc-f038-435b-8ad3-d4bbf0f0fa71.c000.snappy.parquet,part-00000-1e340abc-f038-435b-8ad3-d4bbf0f0fa71.c000.snappy.parquet,930,1777745947000
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/gold/window_aggregations_parquet/part-00000-9660823a-bddb-4829-84c4-f0e8ceeabb6c.c000.snappy.parquet,part-00000-9660823a-bddb-4829-84c4-f0e8ceeabb6c.c000.snappy.parquet,930,1777745938000
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/gold/window_aggregations_parquet/part-00006-fa3f9248-800a-45a3-982e-bfbb1de32156.c000.snappy.parquet,part-00006-fa3f9248-800a-45a3-982e-bfbb1de32156.c000.snappy.parquet,1466,1777745947000
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/gold/window_aggregations_parquet/part-00022-75c2a907-3677-447b-9bbb-655ffff9302a.c000.snappy.parquet,part-00022-75c2a907-3677-447b-9bbb-655ffff9302a.c000.snappy.parquet,1466,1777745948000
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/gold/window_aggregations_parquet/part-00029-1e06b56d-029d-48dc-b44f-e4c663a7cf0f.c000.snappy.parquet,part-00029-1e06b56d-029d-48dc-b44f-e4c663a7cf0f.c000.snappy.parquet,1466,1777745948000
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/gold/window_aggregations_parquet/part-00042-1ca6ad8e-ae59-47bb-8b50-def44888da11.c000.snappy.parquet,part-00042-1ca6ad8e-ae59-47bb-8b50-def44888da11.c000.snappy.parquet,1498,1777745948000
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/gold/window_aggregations_parquet/part-00060-e01fef7b-7477-42eb-b045-38ce751fe1a0.c000.snappy.parquet,part-00060-e01fef7b-7477-42eb-b045-38ce751fe1a0.c000.snappy.parquet,1498,1777745948000
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/gold/window_aggregations_parquet/part-00061-155c0f9a-c846-4e6e-a785-7898303fc617.c000.snappy.parquet,part-00061-155c0f9a-c846-4e6e-a785-7898303fc617.c000.snappy.parquet,1503,1777745948000
dbfs:/Volumes/workspace/stream_processing_pipelines/wikimedia_stream_pipeline/gold/window_aggregations_parquet/part-00067-faf8fa03-04c5-4f28-bfa1-fe41ea072917.c000.snappy.parquet,part-00067-faf8fa03-04c5-4f28-bfa1-fe41ea072917.c000.snappy.parquet,1498,1777745948000


## 12. Proposta de deploy do pipeline

Em um cenário produtivo, este notebook poderia ser convertido em um pipeline operacional no Databricks utilizando **Databricks Jobs**.

A solução poderia ser organizada em dois componentes principais:

1. **Ingestion Job**  
   Responsável por consumir o endpoint Wikimedia Recent Changes e gravar continuamente os eventos brutos na área Raw do Unity Catalog Volume.

2. **Streaming Processing Job**  
   Responsável por executar o processamento com Spark Structured Streaming, aplicando validações, transformações, agregações e gravação das saídas em Parquet.

Uma estrutura possível para operacionalização seria:

```text
wikistream-analytics-pipeline/
│
├── notebooks/
│   └── WikiStream Analytics Pipeline.ipynb
│
├── src/
│   ├── ingest_wikimedia_stream.py
│   └── stream_processing_job.py
│
├── requirements.txt
└── README.md
```

As execuções poderiam ser monitoradas por meio dos logs do Databricks Jobs, enquanto os checkpoints manteriam o controle do progresso incremental do processamento.

Em produção, a ingestão poderia ser executada de forma contínua ou em micro-batches agendados, e o processamento Spark poderia ser configurado com execução recorrente, observabilidade e controle de falhas.

Essa abordagem permite evoluir o notebook acadêmico para uma arquitetura operacional, mantendo separação entre ingestão, processamento, armazenamento, monitoramento e governança.

## 13. Conclusão

Este notebook demonstrou a construção de um pipeline de **stream processing** utilizando Databricks, Apache Spark Structured Streaming e Unity Catalog Volume.

A solução consumiu eventos contínuos do endpoint Wikimedia Recent Changes, armazenou os dados brutos em JSON, processou os registros de forma incremental, aplicou validações, organizou os dados em camadas Raw, Bronze, Silver e Gold, gerou saídas em Parquet e produziu agregações analíticas e temporais.

O pipeline atende aos principais requisitos do enunciado:

| Requisito | Implementação no projeto |
|---|---|
| Construir um Stream Pipeline | Pipeline com ingestão contínua da Wikimedia e processamento com Spark Structured Streaming |
| Utilizar ferramenta ou plataforma | Databricks com Apache Spark Structured Streaming |
| Realizar ingestão dos dados | Consumo do endpoint Wikimedia Recent Changes |
| Gerar output em outro formato | Entrada JSON e saída em Parquet |
| Filtro e/ou validação | Camada Silver com regras de qualidade |
| Agregação dos dados | Camada Gold com indicadores por wiki, tipo e usuário |
| Uso de Window Function | Agregação temporal por wiki com janela de 30 segundos |
| Processo de deploy | Proposta de operacionalização com Databricks Jobs |

Com isso, o projeto demonstra não apenas a execução técnica de um pipeline streaming, mas também uma abordagem estruturada de engenharia de dados, com separação em camadas, rastreabilidade, checkpoints, governança de armazenamento e possibilidade de evolução para deploy operacional.